In [ ]:
import os
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA_DIR = "/Users/Mahfuzaxon/Desktop/fintech_data"

def get_path(filename):
    return os.path.join(DATA_DIR, filename)
#Loading dataset files
train_signals = pd.read_csv(get_path("train_signals.csv"))
test_signals = pd.read_csv(get_path("test_signals.csv"))

train_tx = pd.read_parquet(get_path("train_transactions.parquet"))
test_tx = pd.read_parquet(get_path("test_transactions.parquet"))

def extract_advanced_features(signals_df, tx_df):
    signals_df = signals_df.copy()
    tx_df = tx_df.copy()
    
    signals_df['signal_sanasi'] = pd.to_datetime(signals_df['signal_sanasi'])
    tx_df['tranzaksiya_vaqti'] = pd.to_datetime(tx_df['tranzaksiya_vaqti'])
    
    tx_merged = pd.merge(tx_df, signals_df[['signal_id', 'signal_sanasi']], on='signal_id', how='inner')
    tx_merged['days_before_alert'] = (tx_merged['signal_sanasi'] - tx_merged['tranzaksiya_vaqti']).dt.total_seconds() / 86400.0
    tx_merged['hour'] = tx_merged['tranzaksiya_vaqti'].dt.hour
    tx_merged['is_night'] = tx_merged['hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)
    
    w1 = tx_merged[tx_merged['days_before_alert'] <= 1]
    w3 = tx_merged[tx_merged['days_before_alert'] <= 3]
    w7 = tx_merged[tx_merged['days_before_alert'] <= 7]
    w30 = tx_merged[tx_merged['days_before_alert'] <= 30]
    
    agg_main = tx_merged.groupby('signal_id').agg(
        tx_count=('miqdor_indeksi', 'count'),
        amount_sum=('miqdor_indeksi', 'sum'),
        amount_mean=('miqdor_indeksi', 'mean'),
        amount_max=('miqdor_indeksi', 'max'),
        amount_std=('miqdor_indeksi', 'std'),
        night_tx_ratio=('is_night', 'mean'),
        last_tx_days=('days_before_alert', 'min'),
        tx_time_span=('days_before_alert', lambda x: x.max() - x.min())
    ).reset_index()
    
    agg_1d = w1.groupby('signal_id').agg(tx_cnt_1d=('miqdor_indeksi', 'count'), amt_sum_1d=('miqdor_indeksi', 'sum')).reset_index()
    agg_3d = w3.groupby('signal_id').agg(tx_cnt_3d=('miqdor_indeksi', 'count'), amt_sum_3d=('miqdor_indeksi', 'sum')).reset_index()
    agg_7d = w7.groupby('signal_id').agg(tx_cnt_7d=('miqdor_indeksi', 'count'), amt_sum_7d=('miqdor_indeksi', 'sum')).reset_index()
    agg_30d = w30.groupby('signal_id').agg(tx_cnt_30d=('miqdor_indeksi', 'count'), amt_sum_30d=('miqdor_indeksi', 'sum')).reset_index()
    
    dir_table = pd.crosstab(tx_merged['signal_id'], tx_merged['kirim_chiqim'], values=tx_merged['miqdor_indeksi'], aggfunc='sum', normalize='index').reset_index()
    dir_table.columns = ['signal_id' if c == 'signal_id' else f"dir_ratio_{c}" for c in dir_table.columns]
    
    type_table = pd.crosstab(tx_merged['signal_id'], tx_merged['tranzaksiya_turi'], values=tx_merged['miqdor_indeksi'], aggfunc='sum', normalize='index').reset_index()
    type_table.columns = ['signal_id' if c == 'signal_id' else f"type_ratio_{c}" for c in type_table.columns]
    
    res = pd.merge(signals_df, agg_main, on='signal_id', how='left')
    for table in [agg_1d, agg_3d, agg_7d, agg_30d, dir_table, type_table]:
        res = pd.merge(res, table, on='signal_id', how='left')
        
    res['signal_month'] = res['signal_sanasi'].dt.month
    res['signal_dayofweek'] = res['signal_sanasi'].dt.dayofweek
    
    res['burst_ratio_3d_30d'] = (res['tx_cnt_3d'] + 1) / (res['tx_cnt_30d'] + 1)
    res['amount_ratio_3d_30d'] = (res['amt_sum_3d'] + 1) / (res['amt_sum_30d'] + 1)
    
    return res.fillna(0)

print("Extracting features from transactions...")
train_df = extract_advanced_features(train_signals, train_tx)
test_df = extract_advanced_features(test_signals, test_tx)

feature_cols = [c for c in train_df.columns if c not in ['signal_id', 'signal_sanasi', 'eskalatsiya']]
X = train_df[feature_cols]
y = train_df['eskalatsiya']
X_test = test_df[feature_cols]

print(f"Training LightGBM model on {len(feature_cols)} engineered features...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=-1
    )
    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

cv_auc = roc_auc_score(y, oof_preds)
print(f"Updated Cross-Validation ROC-AUC: {cv_auc:.4f}")

team_name = "team_Aethera.csv" 
out_file = get_path(f"team_{team_name}.csv")
sub = pd.DataFrame({'signal_id': test_df['signal_id'], 'ehtimollik': test_preds})
sub.to_csv(out_file, index=False)
print(f"Saved submission file with {len(sub)} predictions to: {out_file}")